# Notebook 06 — 3D Brain Atrophy Visualisation

**What this notebook does:**
1. Generates a procedural MNI-inspired brain surface (no MRI files needed)
2. Maps real hippocampus volume data from `ADNIMERGE.csv` onto the surface as an atrophy heatmap
3. Renders **side-by-side interactive 3D brains**: baseline vs 24-month for demo patient RID 750
4. Saves a static PNG to `results/visualizations/brain_atrophy_3d.png`

**Prerequisites:** Run `02_lstm_model.ipynb` first to produce `demo_data.json`.  
**No MRI scans required** — the brain shape comes from a procedural ellipsoid with sulcal folds.

**Atrophy regions modelled:**
- Hippocampus (memory — first to shrink in AD)
- Entorhinal cortex (adjacent to hippocampus)
- Ventricles (expand as brain shrinks — shown as surface proxy)
- Frontal & parietal cortex (later-stage atrophy)

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Output paths
OUT_DIR = Path('results/visualizations')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_HTML = OUT_DIR / 'brain_atrophy_3d.html'
OUT_PNG  = OUT_DIR / 'brain_atrophy_3d.png'

print('Imports OK')

In [ ]:
# ── Cell 2: Load ADNIMERGE and extract atrophy data ───────────────────────────
df = pd.read_csv('data/raw/ADNIMERGE.csv')

DEMO_RIDS = [750, 667, 1282, 50, 128]

# Focus on RID 750 for the side-by-side (most complete bl + m24 data)
RID = 750

pat = df[df['RID'] == RID].sort_values('visit_num').reset_index(drop=True)
print(pat[['VISCODE', 'visit_num', 'MMSE', 'Hippocampus', 'DX', 'APOE4']].to_string())

# Baseline (visit 0)
bl  = pat[pat['VISCODE'] == 'bl'].iloc[0]
# 24-month (m24)
m24 = pat[pat['VISCODE'] == 'm24'].iloc[0]

hippo_bl  = float(bl['Hippocampus'])
hippo_m24 = float(m24['Hippocampus'])
mmse_bl   = float(bl['MMSE'])
mmse_m24  = float(m24['MMSE'])
apoe4     = int(bl['APOE4'])

hippo_pct_loss = (hippo_bl - hippo_m24) / hippo_bl * 100

print(f'\nRID {RID} | APOE4={apoe4}')
print(f'Hippocampus : {hippo_bl:.0f} mm³ → {hippo_m24:.0f} mm³  ({hippo_pct_loss:.1f}% loss)')
print(f'MMSE        : {mmse_bl:.0f} → {mmse_m24:.0f}')
print(f'Diagnosis   : {bl["DX"]} → {m24["DX"]}')

In [ ]:
# ── Cell 3: Load REAL brain surface from nilearn (fsaverage5) ────────────────
#
# nilearn ships actual MRI-derived brain meshes.
# fsaverage5 has ~10k vertices — fast to render, looks anatomically correct.
# We use the LEFT hemisphere pial surface (the outer cortical surface).

from nilearn import datasets, surface
import numpy as np

# Download/load fsaverage5 (cached after first run, ~5 MB)
fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')

# Load LEFT hemisphere pial surface (outer cortex)
coords_L, faces_L = surface.load_surf_mesh(fsaverage.pial_left)
coords_R, faces_R = surface.load_surf_mesh(fsaverage.pial_right)

# Stack both hemispheres, offset right hemisphere vertex indices
n_L = len(coords_L)
coords = np.vstack([coords_L, coords_R])
faces  = np.vstack([faces_L, faces_R + n_L])

# Extract x, y, z (RAS coordinates in mm)
bx, by, bz = coords[:, 0], coords[:, 1], coords[:, 2]

print(f'Vertices: {len(coords):,}   Faces: {len(faces):,}')
print(f'x (L-R):   [{bx.min():.0f}, {bx.max():.0f}] = {bx.max()-bx.min():.0f}mm')
print(f'y (A-P):   [{by.min():.0f}, {by.max():.0f}] = {by.max()-by.min():.0f}mm')
print(f'z (height):[{bz.min():.0f}, {bz.max():.0f}] = {bz.max()-bz.min():.0f}mm')
print('Real anatomical brain mesh loaded ✓')


In [ ]:
# ── Cell 4: Region weight maps on real brain vertices ────────────────────────
#
# MNI152 approximate region centres (RAS mm):
#   Hippocampus bilateral:  (±28, -20, -18)
#   Entorhinal:             (±25, -10, -30)
#   Amygdala:               (±24, -4,  -20)
#   Precuneus/parietal:     (  0, -65,  40)
#   Frontal (DLPFC):        (±35,  45,  25)
#   Posterior cingulate:    (  0, -50,  25)

import numpy as np

def gaussian_blob(vx, vy, vz, cx, cy, cz, sx, sy, sz):
    return np.exp(-(
        ((vx - cx)**2) / (2*sx**2) +
        ((vy - cy)**2) / (2*sy**2) +
        ((vz - cz)**2) / (2*sz**2)
    ))

def compute_region_weights(vx, vy, vz):
    # Bilateral = left + right hemisphere blobs
    hippo = (gaussian_blob(vx,vy,vz,  28,-20,-18, 12,12,10) +
             gaussian_blob(vx,vy,vz, -28,-20,-18, 12,12,10))

    entorh = (gaussian_blob(vx,vy,vz,  25,-10,-30, 10,10, 9) +
              gaussian_blob(vx,vy,vz, -25,-10,-30, 10,10, 9))

    amyg   = (gaussian_blob(vx,vy,vz,  24, -4,-20,  9, 9, 9) +
              gaussian_blob(vx,vy,vz, -24, -4,-20,  9, 9, 9))

    parietal  = gaussian_blob(vx,vy,vz,   0,-65, 40, 30,18,18)

    frontal   = (gaussian_blob(vx,vy,vz,  35, 45, 25, 18,18,18) +
                 gaussian_blob(vx,vy,vz, -35, 45, 25, 18,18,18))

    cingulate = gaussian_blob(vx,vy,vz,   0,-50, 25, 10,18,15)

    return dict(hippocampus=hippo, entorhinal=entorh, amygdala=amyg,
                parietal=parietal, frontal=frontal, cingulate=cingulate)

weights = compute_region_weights(bx, by, bz)

print('Region weights on real mesh:')
for k, v in weights.items():
    print(f'  {k:15s}: max={v.max():.3f}  (n_affected={( v>0.1).sum():,})')


In [ ]:
# ── Cell 5: Build atrophy intensity maps ─────────────────────────────────────
import numpy as np

def build_atrophy_map(hippo_vol, mmse_score, weights, hippo_ref=6000.0):
    hippo_severity = np.clip((hippo_ref - hippo_vol) / hippo_ref * 4.5, 0, 1)
    mmse_severity  = np.clip((30 - mmse_score) / 30, 0, 1)

    mtl      = (weights['hippocampus'] * 1.00
              + weights['entorhinal']  * 0.75
              + weights['amygdala']    * 0.55) * hippo_severity

    cortical = (weights['parietal']   * 0.55
              + weights['cingulate']  * 0.45
              + weights['frontal']    * 0.30) * mmse_severity

    atrophy = mtl + cortical
    atrophy = atrophy / (atrophy.max() + 1e-8)
    return atrophy

atr_bl  = build_atrophy_map(hippo_bl,  mmse_bl,  weights)
atr_m24 = build_atrophy_map(hippo_m24, mmse_m24, weights)

print(f'Baseline  atrophy — mean: {atr_bl.mean():.3f}, max: {atr_bl.max():.3f}')
print(f'24-month  atrophy — mean: {atr_m24.mean():.3f}, max: {atr_m24.max():.3f}')


In [ ]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = Path('results/visualizations')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_HTML = OUT_DIR / 'brain_atrophy_3d.html'

from nilearn import datasets, surface

fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
coords_L, faces_L = surface.load_surf_mesh(fsaverage.pial_left)
coords_R, faces_R = surface.load_surf_mesh(fsaverage.pial_right)
sulc_L = surface.load_surf_data(fsaverage.sulc_left)
sulc_R = surface.load_surf_data(fsaverage.sulc_right)

n_L = len(coords_L)
coords = np.vstack([coords_L, coords_R])
faces  = np.vstack([faces_L, faces_R + n_L])
sulc   = np.concatenate([sulc_L, sulc_R])
bx, by, bz = coords[:, 0], coords[:, 1], coords[:, 2]
print(f'Mesh: {len(coords):,} verts, {len(faces):,} faces')

# ── Patient data ──────────────────────────────────────────────────────────────
RID            = 750
hippo_bl       = 5256.0
hippo_m24      = 4836.0
mmse_bl        = 27.0
mmse_m24       = 8.0
apoe4          = 1
hippo_pct_loss = (hippo_bl - hippo_m24) / hippo_bl * 100

# ── Gaussian blobs ────────────────────────────────────────────────────────────
def gaussian_blob(vx, vy, vz, cx, cy, cz, sx, sy, sz):
    return np.exp(-(((vx-cx)**2)/(2*sx**2)+((vy-cy)**2)/(2*sy**2)+((vz-cz)**2)/(2*sz**2)))

def compute_weights(vx, vy, vz):
    return dict(
        hippo    = gaussian_blob(vx,vy,vz,  28,-20,-18,14,14,12) + gaussian_blob(vx,vy,vz,-28,-20,-18,14,14,12),
        entorh   = gaussian_blob(vx,vy,vz,  25,-10,-30,12,12,10) + gaussian_blob(vx,vy,vz,-25,-10,-30,12,12,10),
        amyg     = gaussian_blob(vx,vy,vz,  24, -4,-20,10,10,10) + gaussian_blob(vx,vy,vz,-24, -4,-20,10,10,10),
        parietal = gaussian_blob(vx,vy,vz,   0,-65, 40,32,20,20),
        frontal  = gaussian_blob(vx,vy,vz,  35, 45, 25,20,20,20) + gaussian_blob(vx,vy,vz,-35, 45,25,20,20,20),
        cingulate= gaussian_blob(vx,vy,vz,   0,-50, 25,12,20,17),
    )

weights = compute_weights(bx, by, bz)

def build_atrophy(hippo_vol, mmse_score, weights, hippo_ref=6000.0):
    hs = np.clip((hippo_ref - hippo_vol) / hippo_ref * 5.5, 0, 1)
    ms = np.clip((30 - mmse_score) / 30 * 1.4, 0, 1)
    mtl      = (weights['hippo']*1.0 + weights['entorh']*0.8 + weights['amyg']*0.65) * hs
    cortical = (weights['parietal']*0.7 + weights['cingulate']*0.6 + weights['frontal']*0.45) * ms
    a = mtl + cortical
    return a / (a.max() + 1e-8)

atr_bl  = build_atrophy(hippo_bl,  mmse_bl,  weights)
atr_m24 = build_atrophy(hippo_m24, mmse_m24, weights)

# ── Sulcal-blended intensity ──────────────────────────────────────────────────
sulc_norm = (sulc - sulc.min()) / (sulc.max() - sulc.min())

def combined_intensity(atrophy, sulc_norm, strength=0.85):
    # Sulcal base lives in 0.0–0.35 (blue range only)
    base = sulc_norm * 0.35
    # Atrophy hot signal — only kicks in significantly above 0.18 threshold
    hot  = np.where(atrophy > 0.18, atrophy * strength, atrophy * strength * 0.4)
    # Where atrophy is strong, it takes over; otherwise just sulcal texture
    out  = np.where(hot > 0.20, base * 0.15 + hot, base)
    return np.clip(out, 0, 1)

int_bl  = combined_intensity(atr_bl,  sulc_norm, strength=0.42)
int_m24 = combined_intensity(atr_m24, sulc_norm, strength=0.78)

# ── Colorscale: deep navy dominant, subtle warm glow only in atrophy zones ────
# Most of the brain stays in the blue range (0.0–0.42).
# Warm colours only appear at high atrophy values (0.55+) — like the reference.
COLORSCALE = [
    [0.00, '#020510'],   # deepest navy — sulcal floor (very dark)
    [0.07, '#0a1550'],   # dark indigo
    [0.15, '#102888'],   # rich deep blue
    [0.24, '#1848c0'],   # medium electric blue (healthy cortex)
    [0.33, '#1870e0'],   # brighter blue (gyral crowns) — but not blinding
    [0.42, '#10a8c8'],   # muted teal (early transition, subtle)
    [0.52, '#20c060'],   # muted green — mild atrophy, restrained
    [0.62, '#a8c000'],   # olive-yellow (moderate) — much less saturated than before
    [0.72, '#d89000'],   # warm amber (significant) — toned down gold
    [0.83, '#d04010'],   # burnt orange-red (severe)
    [0.93, '#b01010'],   # deep red (critical)
    [1.00, '#700000'],   # darkest maroon (epicentre)
]

i_tri = faces[:,0]; j_tri = faces[:,1]; k_tri = faces[:,2]

BG      = '#030610'
TEXT    = '#dde8ff'
SUBTEXT = '#6878a0'
CYAN    = '#40c8ff'
CORAL   = '#ff7060'

x_range = bx.max() - bx.min()
GAP = x_range * 1.70   # slightly tighter so brains fill more screen

def solid_brain(vx, vy, vz, intensity, name, showscale=False, x_offset=0):
    """Semi-transparent solid surface with strong specular — the main brain body."""
    return go.Mesh3d(
        x=vx+x_offset, y=vy, z=vz,
        i=i_tri, j=j_tri, k=k_tri,
        intensity=intensity,
        intensitymode='vertex',
        colorscale=COLORSCALE,
        cmin=0, cmax=1,
        opacity=0.88,
        name=name,
        showscale=showscale,
        colorbar=dict(
            title=dict(text='Atrophy<br>severity',
                       font=dict(size=12, color=SUBTEXT, family='Inter, sans-serif')),
            thickness=14, len=0.55,
            tickvals=[0, 0.24, 0.42, 0.62, 0.83, 1.0],
            ticktext=['Healthy','Early','Mild','Moderate','Severe','Critical'],
            tickfont=dict(size=10, color=SUBTEXT, family='Inter, sans-serif'),
            bgcolor='rgba(3,6,16,0.9)', bordercolor='#1a2040', borderwidth=1,
            x=1.02, y=0.5, outlinewidth=0,
        ) if showscale else None,
        hovertemplate=f'<b>{name}</b><br>Atrophy: <b>%{{intensity:.3f}}</b><extra></extra>',
        flatshading=False,
        lighting=dict(
            ambient=0.15,
            diffuse=1.0,
            specular=0.60,
            roughness=0.30,
            fresnel=0.35,
        ),
        lightposition=dict(x=300, y=-80, z=600),
    )

def wireframe_brain(vx, vy, vz, intensity, name, x_offset=0):
    """
    Wireframe overlay — renders the triangle edges as thin glowing lines.
    We sample a subset of edges to avoid overplotting (every 4th face).
    This creates the neural-network-wire look from the reference image.
    """
    # Sample every 4th face to keep it light but visible
    step = 4
    sampled_faces = faces[::step]

    # Build edge segments: each face contributes 3 edges
    # We'll use None-separated scatter3d for efficiency
    xs, ys, zs = [], [], []
    vx_off = vx + x_offset
    for tri in sampled_faces:
        for a, b in [(tri[0],tri[1]), (tri[1],tri[2]), (tri[2],tri[0])]:
            # colour edges by atrophy of their midpoint vertex (brightest vertex)
            xs += [vx_off[a], vx_off[b], None]
            ys += [vy[a],     vy[b],     None]
            zs += [vz[a],     vz[b],     None]

    # Colour the wireframe: mix atrophy signal of the sampled vertices
    # Use a single neon blue-cyan for healthy, yellow-green for atrophy
    # Simple approach: single colour with low opacity
    return go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='lines',
        name=f'{name} (wire)',
        line=dict(
            color='rgba(24, 72, 192, 0.10)',  # darker blue, very faint
            width=0.6,
        ),
        hoverinfo='skip',
        showlegend=False,
    )

def atrophy_wireframe(vx, vy, vz, atrophy, name, x_offset=0, threshold=0.25):
    """
    Brighter wireframe only in atrophy zones — the glowing hot lines in the reference.
    """
    step = 2
    sampled_faces = faces[::step]
    xs, ys, zs, cols = [], [], [], []
    vx_off = vx + x_offset

    for tri in sampled_faces:
        a, b, c = tri
        # Only draw if at least one vertex is in atrophy zone
        max_atr = max(atrophy[a], atrophy[b], atrophy[c])
        if max_atr < threshold:
            continue
        # Intensity drives colour: cyan → yellow → red
        t = min(max_atr, 1.0)
        if t < 0.5:
            r = int(32 + t*2*200); g = int(200 - t*100); b_c = int(255 - t*200)
        else:
            r = int(232); g = int(max(0, 200 - (t-0.5)*2*200)); b_c = 20
        alpha = min(0.6, t * 0.8)
        col = f'rgba({r},{g},{b_c},{alpha:.2f})'
        for a_v, b_v in [(a,b),(b,c),(c,a)]:
            xs += [vx_off[a_v], vx_off[b_v], None]
            ys += [vy[a_v],     vy[b_v],     None]
            zs += [vz[a_v],     vz[b_v],     None]
            cols.append(col)

    if not xs:
        return None

    return go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='lines',
        name=f'{name} (atrophy wire)',
        line=dict(color='rgba(200, 140, 0, 0.22)', width=1.0),
        hoverinfo='skip',
        showlegend=False,
    )

# Build all traces
trace_bl_solid  = solid_brain(bx, by, bz, int_bl,  'Baseline',  showscale=False, x_offset=0)
trace_m24_solid = solid_brain(bx, by, bz, int_m24, '24 months', showscale=True,  x_offset=GAP)
trace_bl_wire   = wireframe_brain(bx, by, bz, int_bl,  'Baseline',  x_offset=0)
trace_m24_wire  = wireframe_brain(bx, by, bz, int_m24, '24 months', x_offset=GAP)
trace_bl_hot    = atrophy_wireframe(bx, by, bz, atr_bl,  'Baseline',  x_offset=0,   threshold=0.45)
trace_m24_hot   = atrophy_wireframe(bx, by, bz, atr_m24, '24 months', x_offset=GAP, threshold=0.32)

all_traces = [trace_bl_solid, trace_m24_solid,
              trace_bl_wire,  trace_m24_wire]
if trace_bl_hot:  all_traces.append(trace_bl_hot)
if trace_m24_hot: all_traces.append(trace_m24_hot)

fig = go.Figure(data=all_traces)

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor=BG,
        camera=dict(
            eye=dict(x=0.0, y=-1.75, z=0.50),   # closer = bigger brains
            up=dict(x=0, y=0, z=1),
            center=dict(x=0.25, y=0, z=-0.02),
        ),
        aspectmode='data',
    ),
    title=dict(
        text=(
            f'<b>3D Brain Atrophy — Patient RID {RID}</b>'
            f'<br><span style="font-size:12px;color:{SUBTEXT}">'
            f'APOEε4={apoe4}  ·  Hippocampus loss {hippo_pct_loss:.1f}% over 24 months'
            f'  ·  MMSE {int(mmse_bl)} → {int(mmse_m24)}/30'
            f'</span>'
        ),
        font=dict(size=18, color=TEXT, family='Georgia, serif'),
        x=0.47, xanchor='center', y=0.97,
    ),
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    margin=dict(l=20, r=155, t=115, b=120),
    height=730, width=1360,
    font=dict(family='Inter, DM Sans, sans-serif', color=TEXT),
    showlegend=False,
    annotations=[
        dict(
            text=(
                f'<b style="font-size:15px;color:{CYAN}">Baseline</b><br>'
                f'<span style="font-size:11px;color:{SUBTEXT}">MMSE '
                f'<b style="color:{TEXT}">{int(mmse_bl)}</b>/30  ·  '
                f'Hippocampus <b style="color:{TEXT}">{hippo_bl:,.0f}</b> mm³</span>'
            ),
            xref='paper', yref='paper', x=0.18, y=1.035,
            showarrow=False, font=dict(size=14, family='Georgia, serif'),
            xanchor='center', align='center',
        ),
        dict(
            text=(
                f'<b style="font-size:15px;color:{CORAL}">24 months</b><br>'
                f'<span style="font-size:11px;color:{SUBTEXT}">MMSE '
                f'<b style="color:{TEXT}">{int(mmse_m24)}</b>/30  ·  '
                f'Hippocampus <b style="color:{TEXT}">{hippo_m24:,.0f}</b> mm³</span>'
            ),
            xref='paper', yref='paper', x=0.75, y=1.035,
            showarrow=False, font=dict(size=14, family='Georgia, serif'),
            xanchor='center', align='center',
        ),
        dict(
            text=(
                f'<span style="color:{CORAL}">●</span> '
                '<span style="color:#c0cce8">Hippocampus · Entorhinal · Amygdala</span>'
                f'  <span style="color:{SUBTEXT}">— medial temporal (first affected)</span>'
            ),
            xref='paper', yref='paper', x=0.47, y=-0.065,
            showarrow=False, font=dict(size=11, family='Inter, sans-serif'),
            xanchor='center',
        ),
        dict(
            text=(
                '<span style="color:#c8e800">●</span> '
                '<span style="color:#c0cce8">Parietal · Cingulate · Frontal</span>'
                f'  <span style="color:{SUBTEXT}">— later-stage cortical spread</span>'
            ),
            xref='paper', yref='paper', x=0.47, y=-0.125,
            showarrow=False, font=dict(size=11, family='Inter, sans-serif'),
            xanchor='center',
        ),
        dict(
            text=f'<span style="color:#2a3050">🖱 Drag to rotate  ·  Scroll to zoom  ·  Double-click to reset</span>',
            xref='paper', yref='paper', x=0.47, y=-0.185,
            showarrow=False, font=dict(size=10, family='Inter, sans-serif'),
            xanchor='center',
        ),
    ],
)

fig.write_html(str(OUT_HTML), include_plotlyjs='cdn')
print(f'✓ Saved → {OUT_HTML}')


In [ ]:
# ── Cell 7: Save static PNG (kaleido fix) ────────────────────────────────────
# If you get a kaleido error, run:  pip install -U kaleido
# Then RESTART the kernel and re-run from Cell 1.

import subprocess, sys

def ensure_kaleido():
    try:
        import kaleido
        return True
    except ImportError:
        print('kaleido not found — installing...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'kaleido', '-q'])
        print('kaleido installed. Restart kernel if this is the first install.')
        return False

ensure_kaleido()

try:
    fig.write_image(str(OUT_PNG), width=1100, height=560, scale=2)
    print(f'✓ Saved static PNG  → {OUT_PNG}')
    print(f'  This PNG will auto-appear in the Dash gallery.')
except Exception as e:
    print(f'PNG export skipped: {e}')
    print(f'  Workaround: open {OUT_HTML} in Chrome and use File → Print → Save as PDF,')
    print(f'  or take a screenshot and save as brain_atrophy_3d.png in results/visualizations/')


In [ ]:
# ── Cell 8: All 5 demo patients — atrophy comparison bar summary ──────────────
# Shows hippocampus % loss for each patient alongside the brain renders.
import plotly.graph_objects as go

summary_rows = []
for rid in DEMO_RIDS:
    p = df[df['RID'] == rid].sort_values('visit_num')
    bl_row  = p[p['VISCODE'] == 'bl']
    m24_row = p[p['VISCODE'] == 'm24']
    if bl_row.empty or m24_row.empty:
        continue
    h_bl  = bl_row.iloc[0]['Hippocampus']
    h_m24 = m24_row.iloc[0]['Hippocampus']
    if pd.isna(h_bl) or pd.isna(h_m24):
        continue
    pct = (h_bl - h_m24) / h_bl * 100
    summary_rows.append({
        'RID': rid,
        'Hippo BL': h_bl,
        'Hippo M24': h_m24,
        'Loss %': round(pct, 2),
        'MMSE BL':  bl_row.iloc[0]['MMSE'],
        'MMSE M24': m24_row.iloc[0]['MMSE'],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

colors = ['#ef4444' if r > 5 else '#f59e0b' if r > 2 else '#22c55e' 
          for r in summary_df['Loss %']]

bar_fig = go.Figure(go.Bar(
    x=[f'RID {r}' for r in summary_df['RID']],
    y=summary_df['Loss %'],
    marker_color=colors,
    text=[f"{v:.1f}%" for v in summary_df['Loss %']],
    textposition='outside',
))
bar_fig.update_layout(
    title='Hippocampus volume loss: baseline → 24 months (all demo patients)',
    yaxis_title='Volume loss (%)',
    paper_bgcolor='rgba(245,245,248,1)',
    plot_bgcolor='rgba(245,245,248,1)',
    height=380,
)
bar_fig.show()

In [ ]:
# ── Cell 9: Add brain_atrophy_3d to the Dash app gallery ─────────────────────
# The Dash app (app.py) serves PNGs from results/visualizations/.
# If you exported brain_atrophy_3d.png above, it will auto-appear in the gallery.
#
# To also embed the INTERACTIVE HTML version, add this snippet to app.py:
#
#   html.Iframe(
#       src='/hub-results/visualizations/brain_atrophy_3d.html',
#       style={'width': '100%', 'height': '580px', 'border': 'none'}
#   )
#
# And register the .html extension in the hub_results() route:
#   if not candidate.is_file() or candidate.suffix.lower() not in ('.png', '.html'):

print('Notebook complete!')
print(f'Outputs:')
print(f'  Interactive: {OUT_HTML}')
print(f'  Static PNG:  {OUT_PNG}  (requires kaleido)')
print()
print('To embed the interactive brain in app.py, see the comment in this cell.')